<a href="https://colab.research.google.com/github/gilIolgenblum/CrowdingModeWorkshop/blob/main/tutorials/01_plug_and_play_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Plug-and-Play Simulation

## Goal
In this tutorial, you will learn how to run a forward simulation of the crowding model. A "forward simulation" means that we already know the protein parameters and the cosolute (crowder) interaction parameters, and we simply want to calculate how the protein's stability changes as a function of the cosolute concentration.

## Setup
We begin by importing the `crowding` package and `matplotlib.pyplot` for visualization.

In [ ]:
import os
import sys

# Check if we are running in Google Colab
if 'google.colab' in str(get_ipython()):
    # Clone the repository to get the data files
    !git clone https://github.com/gilIolgenblum/CrowdingModeWorkshop.git
    # Change the working directory to the repository root
    os.chdir('/content/CrowdingModeWorkshop')
    # Install the package dependencies
    !pip install -e .

    # TELL PYTHON WHERE THE SOURCE FOLDER IS
    sys.path.append('/content/CrowdingModeWorkshop/src')

In [ ]:
import crowding as cr
import matplotlib.pyplot as plt
import numpy as np

print(f"Module imported successfully: {cr.__name__}")

## Define the Protein and Cosolute
The model requires:
1. **Protein properties:** Primarily its change in Solvent Accessible Surface Area (SASA) in $A^2$.
2. **Cosolute properties:** The excluded volume parameter ($\nu$), and the Flory-Huggins nonideal interaction parameters ($\chi$ and $\chi_{TS}$).

For this example use the valeus for MET16, SASA=419.0, and for glycerol, $\nu=2.479$, $\chi=0.610$, and $\chi_{TS}=-3.650$. 
Let's define a hypothetical small protein and a standard cosolute.

In [ ]:
# Define the protein
MET16 = cr.Protein(SASA=???)

# Define the cosolute
glycerol = cr.Cosolute(
    nu=???,      # Excluded volume parameter
    chi=???,     # Flory-Huggins interaction parameter
    chiTS=???   # Entropic component of chi
)

In [ ]:
print(MET16, glycerol, sep='\n')

## Build and Solve the Model
Now, we construct the `CrowdingModel`. Since this is a forward simulation, we will arbitrarily set the soft interaction parameters $\varepsilon$ and $\varepsilon_{TS}$ to non-zero values to see their effect.

We will simulate the concentration range up to $\phi = 0.15$ with a step size of $\Delta \phi=0.0001$.

We then use the `CrowdingModel` object to solve the equilibrium condition describing the parition of cosolutes between the protein and bulk domain:
$$
\nu\mu_S^\mathrm{bulk}-\mu_C^\mathrm{bulk}+\mu_C^\mathrm{surf}-\nu\mu_S^\mathrm{surf}=0
$$

This is done using the `solve_equil` function:
![Solve Equil Workflow](assets/solve_equil_flowchart.svg)

In [ ]:
# Initialize the binary model
model = cr.CrowdingModel(
    protein=???,
    cosolute=???,
    eps=???,        # Soft interactions parameter
    epsTS=???,       # Entropic component of soft interactions
    phiC_max=???,    # Maximum volume fraction to simulate
    dphiC=???,      # Step size for the grid
    T=298.15          # Temperature in Kelvin
)

# Solve the thermodynamic equilibrium equations across the concentration grid
model.solve_equil(progress_bar=True)

## Extract and Plot Results
The `CrowdingModel` can seamlessly convert all its internal arrays into a `pandas.DataFrame` for easy inspection and plotting.

In [ ]:
# Convert results to DataFrame
model.to_pandas()
results = model.results

# Display the first few rows
display(results.head())

### Understanding the Results Table

The `results` DataFrame contains a comprehensive thermodynamic profile of the simulated system at each concentration step. Here is a breakdown of the most important columns:

#### Concentrations and Chemical Potentials
- `phiC`, `phiS`: The bulk volume fractions of the Cosolute and Solvent, respectively.
- `phiCsurf`, `phiSsurf`: The volume fractions of the Cosolute and Solvent *inside the protein domain* (the local concentration).
- `molar`, `molal`, `osm`: The bulk cosolute concentration expressed in molarity (M), molality (mol/kg), and osmolality.
- `gamma`, `gammaC`: The solvation and preferential interaction coefficients.

#### Thermodynamic Components (in $k_B T$)
These columns represent the free energy changes ($\Delta\Delta A^0$ or $\Delta\Delta G^0$), enthalpy changes ($\Delta\Delta E^0$ or $\Delta\Delta H^0$), and entropy changes ($T\Delta\Delta S^0$) broken down by their physical origin:
- `*_nu`: The contribution from **excluded volume** (steric repulsions).
- `*_chi`: The contribution from **Flory-Huggins** non-ideal solvent interactions.
- `*_eps`: The contribution from **soft interactions** between the protein and cosolute.
- `ddA`, `ddE`, `TddS`: The **total** sum of the components for free energy, enthalpy, and entropy.

#### Thermodynamic Components (in kJ/mol)
All the thermodynamic columns listed above are also provided in standard chemical units (kJ/mol), indicated by the `_kJ` suffix (e.g., `ddA_kJ`, `ddA_nu_kJ`, `TddS_kJ`). Otherwise, thermodynamic potentials are given in $k_B T$.

## Exporting Results
You can export the results DataFrame to a CSV file for your own records or for plotting in other software (e.g., Origin, Prism).

In [ ]:
# Export to CSV (commented out to prevent writing during the tutorial)
# results.to_csv("binary_simulation_results.csv", index=False)
print("Simulation complete! You can now analyze the `results` dataframe.")

# Binary Model: Plotting Showcase

This notebook demonstrates the plotting functionality available in `Plotter` object for analyzing and visualizing the thermodynamics of the crowding model.

Start by generating a plotter object:

In [ ]:
plotter = cr.Plotter(model)

## Comprehensive Results Plot

The `plot_results()` method generates a 3x3 grid of subplots covering all critical thermodynamic properties. By default, the x-axis is volume fraction ($\phi$), but you can change it using `concentration_type='molal'` or `concentration_type='molar'`. 

**Try changing the concentration type!**

In [ ]:
fig = plotter.plot_results()
plt.show()

## Individual Plots
The `Plotter` also provides individual plotting methods.

### Change in the Preferential Interaction Parameter ($\Delta\Gamma_S=\Gamma_{S,\mathrm{native}}-\Gamma_{S,\mathrm{denatured}}$)

In [ ]:
fig = plotter.plot_gamma(concentration_type='molal')
plt.show()

### Folding Free Energy ($\Delta\Delta G^0$)

In [ ]:
fig = plotter.plot_ddG(concentration_type='molal')
plt.show()

### Folding Enthalpy ($\Delta\Delta H^0$)

In [ ]:
fig = plotter.plot_ddH(concentration_type='molal')
plt.show()

### Folding Entropy ($T\Delta\Delta S^0$)

In [ ]:
fig = plotter.plot_TddS(concentration_type='molal')
plt.show()

### Chemical Potential ($\mu$)

In [ ]:
fig = plotter.plot_mu(concentration_type='molal')
plt.show()

### Osmotic Pressure ($\Pi$)

In [ ]:
fig = plotter.plot_osm(concentration_type='molal')
plt.show()

### Enthalpy-Entropy Compensation (Standalone)

In [ ]:
fig = plotter.plot_EEC()
plt.show()